In [9]:
# add root to path
import sys; sys.path.append('..')

import transformers
from training_utils import make_supervised_data_module
from collections import namedtuple
from typing import Dict, List

import torch
from tqdm import tqdm
from eval_utils import load_data, extract_decision, load_model, get_model_generations, get_eval_accuracy, get_eval_loss

from peft import AutoPeftModelForCausalLM

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = '/root/MoRA/pub-med-qa/lora_rank128_lr1e-4_witheval/checkpoint-400'
print("Loading model...")
model = load_model(ckpt_path)
model.to(device)

In [11]:
# Load pubmed test set
import json
import uuid


data_path = 'qiaojin/PubMedQA'
# subset = 'pqa_labeled'
subset = 'pqa_artificial'

# Load the data using the new function
data_module, tokenizer = load_data(
    data_path=data_path,
    base_model='meta-llama/Llama-3.2-1B-Instruct',
    subset=subset,
    data_length=1000, # for testing
)

# List[Dict[str, str]] 
# Array with num_samples elements. Each element has two fields, input_ids and labels.
data = data_module['train_dataset'] 
# print(data[0]['labels'])

# Does tokenization and padding
collator = data_module['data_collator']

# Dict[str, torch.Tensor] with keys input_ids, labels, attention_mask
# tokenized_data = collator(data)

# One of ['yes', 'no', 'maybe']
decisions_y = [extract_decision(d['labels']) for d in data] 
decisions_y

# Save to file
# with open("pub-med-eval/true_decisions_y.json", "w") as f:
    # json.dump(decisions_y, f)

(SOURCES LOG) <|start_header_id|>system<|end_header_id|>

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request. <|eot_id|><|start_header_id|>user<|end_header_id|>

### Instruction:
Is helicobacter pylori infection associated with milder gastro-oesophageal reflux disease?

### Input:
We have previously demonstrated a negative relationship between the prevalence of Helicobacter pylori and gastro-oesophageal reflux disease (GERD).

---

To study the effects of H. pylori infection on the severity of GERD.

---

Ethnic Chinese patients with frequent heartburn and/or endoscopic oesophagitis were studied. Endoscopic examination was performed to assess the severity of oesophagitis (modified Savary-Miller grading) and the presence of hiatus hernia. Biopsies were taken for rapid urease testing and confirmation of Barrett's oesophagus. Risk factors which may affect the severity of oesophagitis 

['yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',


In [ ]:
decisions_y

In [ ]:


# Setup input prompts
prompts = [data[i]['input_ids'] for i in range(len(data))]
# print(prompts[0])

# Get model generations
generations = get_model_generations(model, prompts)
generations_str = tokenizer.batch_decode(generations, skip_special_tokens=True)

decisions_yhat = [extract_decision(g) for g in generations_str]
eval_accuracy = get_eval_accuracy(decisions_yhat, decisions_y)

# Save all results
results = {
    'decisions_yhat': decisions_yhat,
    'decisions_y': decisions_y,
    'generations_str': generations_str,
    'eval_accuracy': eval_accuracy,
    'data_path': data_path,
    'subset': subset,
    'ckpt_path': ckpt_path,
}

# Save results to json
id = str(uuid.uuid4())[:8]
with open(f'pub-med-eval/{subset}_results_{id}.json', 'w') as f:
    json.dump(results, f)

In [54]:
# # Labels and Input Ids have same shape. DO NOT USE FOR GENERATION!
# print(tokenizer.decode(tokenized_data['input_ids'][1], skip_special_tokens=True))
# # print(tokenizer.decode(tokenized_data['labels'][1], skip_special_tokens=True)) # Can't be decoded because of -100!
# print(tokenized_data['input_ids'][1].shape, tokenized_data['labels'][1].shape)

# Labels starts with a bunch of -100s which is the IGNORE_INDEX
# because the input ids locations are not considered part of the loss
# The loss is only computed for the labels that are not -100, which are in the middle.
# Finally there is a bunch of -100s at the end for padding but ignored for loss.

# Input ids meanwhile starts with the token ids (no left ignore) and ends with padding tokens (2).

In [57]:
# E2E generation example from string -> tokenized -> model -> decoded output
# test_input = tokenizer.encode("Hello, how are you?", return_tensors="pt").to(device)
# print(tokenizer.batch_decode(model.generate(test_input, max_new_tokens=128).cpu(), pad_token_id=2))

In [ ]:
# Loss values are not matching the training log loss values
# eval_loss, eval_loss_by_batch = get_eval_loss(model, tokenized_data)
# print(eval_loss)